# RDLNet 两阶段训练（Colab 最小可跑）

1. **挂载 Google Drive**：checkpoint 与数据放在 Drive，断线不丢。  
2. **拉代码 / 装依赖**（二选一：git clone 或上传 zip 解压到 `/content/doc-scan`）。  
3. **阶段 1 蒸馏** → 权重存 Drive。  
4. **阶段 2 RDLNet**：`--resume` 续跑；`--save-every-steps` 防中途断线。

把下面路径里的 `你的文件夹` 改成自己的 Drive 目录即可。

In [ ]:
# @title 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. 路径与依赖（改 DRIVE_ROOT 即可）
import os, sys, subprocess

# 所有数据与权重放在 Drive 下同一父目录，例如：MyDrive/RDLNet_work
DRIVE_ROOT = '/content/drive/MyDrive/RDLNet_work'  # <-- 改成你的目录
REPO_DIR = os.path.join(DRIVE_ROOT, 'doc-scan')    # 代码解压或 clone 到这里

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.chdir('/content')

# 若尚未有代码：任选其一
# A) git clone（需有仓库 URL）
# !git clone https://github.com/YOU/doc-scan.git "{REPO_DIR}"
# B) 本地上传 doc-scan.zip 到 DRIVE_ROOT 后解压：
# !unzip -q "{os.path.join(DRIVE_ROOT, 'doc-scan.zip')}" -d "{DRIVE_ROOT}"

assert os.path.isdir(REPO_DIR), f'请先让 REPO_DIR 存在: {REPO_DIR}'
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

req = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.isfile(req):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
seg = os.path.join(REPO_DIR, 'segment-anything')
if os.path.isdir(seg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', seg])
print('OK cwd=', os.getcwd())

In [ ]:
# @title 3. 阶段 1：Light-SAM 蒸馏（断线后用 --resume 同一路径）
import subprocess, sys, os

IMAGE_DIR = os.path.join(DRIVE_ROOT, 'images_unlabeled')   # 无标注图片文件夹
TEACHER = os.path.join(DRIVE_ROOT, 'sam_vit_h_4b8939.pth')
OUT = os.path.join(DRIVE_ROOT, 'checkpoints', 'distill_stage1.pt')
os.makedirs(os.path.dirname(OUT), exist_ok=True)

cmd = [
    sys.executable, 'train_distill.py',
    '--image-dir', IMAGE_DIR,
    '--teacher-checkpoint', TEACHER,
    '--output', OUT,
    '--epochs', '1',
    '--batch-size', '1',
    '--img-size', '1024',
    '--save-every-steps', '200',
    # 续跑时取消下面注释并指向已有 pt
    # '--resume', OUT,
    # '--epochs', '10',
]
subprocess.check_call(cmd, cwd=REPO_DIR)

In [ ]:
# @title 4. 阶段 2：RDLNet（需 annotations.json；骨干用阶段 1 权重）
import subprocess, sys, os

ANN = os.path.join(DRIVE_ROOT, 'annotations.json')
IMG_ROOT = DRIVE_ROOT  # JSON 里 file_name / mask 相对此根目录
DISTILL_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'distill_stage1.pt')
OUT2 = os.path.join(DRIVE_ROOT, 'checkpoints', 'rdlnet.pt')

cmd = [
    sys.executable, 'train_rdlnet.py',
    '--annotations', ANN,
    '--image-root', IMG_ROOT,
    '--distill-checkpoint', DISTILL_CKPT,
    '--output', OUT2,
    '--epochs', '5',
    '--batch-size', '1',
    '--img-size', '1024',
    '--save-every-steps', '100',
    # 续跑：
    # '--resume', OUT2,
    # '--epochs', '20',
]
subprocess.check_call(cmd, cwd=REPO_DIR)

## 断线后续跑

- **阶段 1**：把 `train_distill.py` 加上 `--resume` 指向 `distill_stage1.pt`（或 `distill_stage1_latest.pt`），`--epochs` 写「还要训几轮」。  
- **阶段 2**：`--resume` 指向 `rdlnet.pt` 或 `rdlnet_latest.pt`，**不要**再带 `--distill-checkpoint`。

显存不够时先把 `--img-size` 改为 `512`，阶段 1 与 2 必须一致。